<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 2 - Neural Networks</font><br>
<font size=20 color=#00c382>Bank Data</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**

https://archive.ics.uci.edu/dataset/222/bank+marketing

---

# Initialise Notebook

In [38]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [39]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
from matplotlib.pyplot import figure
import matplotlib.ticker as mticker
import matplotlib.animation as animation
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# IPython Display 
from IPython.display import HTML

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [40]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [41]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [42]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [43]:
internet_required = False

In [44]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Load and Explore the Dataset

In [45]:
if online:
    # Fetch the dataset from the Internet
    df = pd.read_csv('bank-full.csv', delimiter=';')
else:
    # Load the dataset from local file
    df = pd.read_csv('bank-full.csv', delimiter=';')

### Explore the dataset

In [46]:
print (df.columns)
print ("")
print (df.shape)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

(45211, 17)


In [47]:
df.sample(20)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
30604,33,management,divorced,tertiary,no,387,no,no,cellular,6,feb,17,5,-1,0,unknown,no
10139,49,entrepreneur,married,tertiary,no,384,no,no,unknown,11,jun,64,1,-1,0,unknown,no
11742,54,blue-collar,married,primary,no,2281,yes,no,unknown,20,jun,158,1,-1,0,unknown,no
31140,56,management,divorced,tertiary,no,722,no,no,cellular,19,feb,134,3,94,2,failure,no
32249,35,services,single,secondary,no,1063,yes,yes,cellular,16,apr,327,1,-1,0,unknown,no
17873,46,housemaid,married,secondary,no,271,yes,no,cellular,30,jul,1013,29,-1,0,unknown,yes
17773,28,technician,married,tertiary,no,107,yes,no,cellular,29,jul,122,13,-1,0,unknown,no
21649,37,technician,single,tertiary,no,971,no,no,cellular,19,aug,120,6,-1,0,unknown,no
10563,38,technician,married,tertiary,no,606,no,no,unknown,16,jun,193,2,-1,0,unknown,no
35390,54,blue-collar,married,primary,no,6242,yes,no,cellular,7,may,162,1,-1,0,unknown,no


In [48]:
df.dropna(inplace=True)
df.shape

(45211, 17)

## Conversion and Encoding Ready for the MLP

xscaled​=max(x)−min(x)x−min(x)​

### Convert Binary Variables

In [49]:
# Convert variable 'y' to binary
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Convert 'default'
df['default'] = df['default'].map({'yes': 1, 'no': 0})

# Convert 'housing'
df['housing'] = df['housing'].map({'yes': 1, 'no': 0})

# COnvert 'loan'
df['loan'] = df['loan'].map({'yes': 1, 'no': 0})

In [50]:
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
5677,50,services,married,secondary,0,683,1,1,unknown,26,may,129,3,-1,0,unknown,0
15782,33,student,married,secondary,0,0,0,0,cellular,21,jul,100,1,-1,0,unknown,0
16192,27,blue-collar,married,primary,0,300,0,1,cellular,22,jul,209,2,-1,0,unknown,0
10078,36,technician,married,tertiary,0,4139,0,0,unknown,11,jun,175,4,-1,0,unknown,0
44259,39,blue-collar,married,primary,0,558,1,0,unknown,22,jul,23,1,531,3,failure,0
13909,59,blue-collar,divorced,unknown,0,454,1,0,cellular,10,jul,293,1,-1,0,unknown,0
16747,36,management,married,tertiary,0,268,1,1,cellular,24,jul,823,1,-1,0,unknown,0
33060,33,blue-collar,married,primary,0,-3,1,0,telephone,17,apr,114,2,-1,0,unknown,0
19196,36,management,single,tertiary,0,546,1,0,cellular,6,aug,91,3,-1,0,unknown,0
40885,47,technician,married,tertiary,0,2408,0,0,cellular,12,aug,236,1,104,1,success,0


In [51]:
df['loan'].unique()

array([0, 1])

This from Gemini - why? Need to understand this and reimplement 

In [52]:
sys.exit('Stopping notebook run here.')    

SystemExit: Stopping notebook run here.

In [ ]:


# Separate features and target
X = df.drop('y', axis=1)
y = df['y'].values.reshape(-1, 1) # Reshape for matrix multiplication

# One-Hot Encoding for categorical variables
# This converts 'job', 'marital', etc., into multiple numeric columns
X_encoded = pd.get_dummies(X).astype(float)

# Feature Scaling (Min-Max Scaling)
# Neural networks hate large values like 'balance' mixed with small ones like 'day'
X_min = X_encoded.min()
X_max = X_encoded.max()
X_scaled = (X_encoded - X_min) / (X_max - X_min)

# Convert to NumPy array
X_final = X_scaled.to_numpy()

print(f"Original features: {X.shape[1]}")
print(f"Input neurons needed: {X_final.shape[1]}")

Original features: 16
Input neurons needed: 48


---

# Data Cleaning


...do stuff...

### Write out the cleaned dataset for further use



In [ ]:
# Create filenames, one timestamped, one 'latest'
filepath = "."
filename = f'{filepath}/bank_full_cleaned_{now_date_timestamp}.csv'
latest_filename = f'{filepath}/bank_full_cleaned_latest.csv'

# Write out CSV files
df.to_csv(filename, index=False)
df.to_csv(latest_filename, index=False)